In [2]:
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms

def flatten_bchw_to_bchw_linear(tensor_4d):
    """
    Flattens a BCHW tensor (B, C, H, W) to B x (C*H*W) using explicit index arithmetic.
    Avoids dtype mismatch between PyTorch and NumPy.
    """
    # Convert PyTorch tensor to NumPy array upfront if needed
    data = tensor_4d.numpy() if isinstance(tensor_4d, torch.Tensor) else tensor_4d

    B, C, H, W = data.shape
    CHW = C * H * W

    # Initialize output array using NumPy's dtype
    flattened = np.zeros((B, CHW), dtype=data.dtype)

    for b in range(B):
        for c in range(C):
            for h in range(H):
                for w in range(W):
                    # Explicit index generation for row-major layout
                    k = c * (H * W) + h * W + w
                    flattened[b, k] = data[b, c, h, w]

    return flattened


def reconstruct_bchw_from_linear(flattened, target_shape):
    """
    Reconstructs a B x (C*H*W) tensor back to (B, C, H, W) using inverse index arithmetic.
    """
    B, C, H, W = target_shape
    HW = H * W
    CHW = C * HW

    reconstructed = np.zeros((B, C, H, W), dtype=flattened.dtype)

    for b in range(B):
        for k in range(CHW):
            c = k // HW
            r = k % HW
            h = r // W
            w = r % W
            reconstructed[b, c, h, w] = flattened[b, k]

    return reconstructed

def compute_errors(original, reconstructed):
    """Computes E_max and MAE between original and reconstructed tensors."""
    orig_np = original.numpy() if isinstance(original, torch.Tensor) else original
    diff = np.abs(orig_np - reconstructed)

    e_max = np.max(diff)
    mae = np.mean(diff)
    return e_max, mae

def run_experiments():
    print("=" * 75)
    print(f"{'Dataset / Input':<18} | {'B':<3} | {'C':<4} | {'H':<3} | {'W':<3} | {'E_max':<8} | {'MAE':<8}")
    print("=" * 75)

    experiments = []

    # 1. MNIST Sample
    try:
        mnist = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
        mnist_img = mnist[0][0].unsqueeze(0)  # Shape: (1, 1, 28, 28)
        experiments.append(("MNIST/sample", mnist_img))
    except Exception:
        experiments.append(("MNIST/sample (synth)", torch.randn(1, 1, 28, 28)))

    # 2. CIFAR-10 Sample
    try:
        cifar = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms.ToTensor())
        cifar_img = cifar[0][0].unsqueeze(0)  # Shape: (1, 3, 32, 32)
        experiments.append(("CIFAR/sample", cifar_img))
    except Exception:
        experiments.append(("CIFAR/sample (synth)", torch.randn(1, 3, 32, 32)))

    # 3. Synthetic Feature Maps with varying channel counts
    channels = [8, 16, 32, 64, 128, 256, 500]
    for idx, c in enumerate(channels, 1):
        # Using smaller H, W for large channel counts to keep explicit loops execution time fast
        h, w = (14, 14) if c <= 128 else (7, 7)
        synth_tensor = torch.randn(2, c, h, w)
        experiments.append((f"Synthetic fmap{idx}", synth_tensor))

    # Execute and output results table
    for name, tensor in experiments:
        B, C, H, W = tensor.shape

        # Core transformation algorithms
        flat = flatten_bchw_to_bchw_linear(tensor)
        reconstructed = reconstruct_bchw_from_linear(flat, (B, C, H, W))

        # Optional independent verification using PyTorch view/reshape
        ref_flat = tensor.view(B, -1).numpy()
        assert np.allclose(flat, ref_flat), f"Verification failed for {name}!"

        # Compute errors
        e_max, mae = compute_errors(tensor, reconstructed)

        print(f"{name:<18} | {B:<3} | {C:<4} | {H:<3} | {W:<3} | {e_max:<8.1e} | {mae:<8.1e}")

if __name__ == "__main__":
    run_experiments()

Dataset / Input    | B   | C    | H   | W   | E_max    | MAE     
MNIST/sample       | 1   | 1    | 28  | 28  | 0.0e+00  | 0.0e+00 
CIFAR/sample       | 1   | 3    | 32  | 32  | 0.0e+00  | 0.0e+00 
Synthetic fmap1    | 2   | 8    | 14  | 14  | 0.0e+00  | 0.0e+00 
Synthetic fmap2    | 2   | 16   | 14  | 14  | 0.0e+00  | 0.0e+00 
Synthetic fmap3    | 2   | 32   | 14  | 14  | 0.0e+00  | 0.0e+00 
Synthetic fmap4    | 2   | 64   | 14  | 14  | 0.0e+00  | 0.0e+00 
Synthetic fmap5    | 2   | 128  | 14  | 14  | 0.0e+00  | 0.0e+00 
Synthetic fmap6    | 2   | 256  | 7   | 7   | 0.0e+00  | 0.0e+00 
Synthetic fmap7    | 2   | 500  | 7   | 7   | 0.0e+00  | 0.0e+00 
